<a href="https://colab.research.google.com/github/phong6786789/OMNIVOICE-MOD/blob/main/OmniVoice_CLONE_PhongSubi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phong Subi - VOICEOMNI MOD

> **GPU:** T4 16GB (tự động) · **Thời gian:** ~3 phút lần đầu, ~30s/lần sau

## Hướng dẫn

1. **Runtime** → **Run all** (Ctrl+F9)
2. Nhập văn bản → Click **Tạo giọng nói**
3. Nghe + tải file WAV về

> Powered by OmniVoice (MIT License) — github.com/k2-fsa/OmniVoice


In [1]:
print('Dang cai dat (~1 phut)...')
!pip install -q omnivoice gradio "numpy<2.1" "requests==2.32.4"
print('Cai dat hoan tat!')

Dang cai dat (~1 phut)...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.5/168.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 4.1 MB/s eta 0:00:00
Cai dat hoan tat!


In [2]:
print("🚀 Đang khởi động Phong Subi - OMNIVOICE MOD...")


# ============================================================
# IMPORT
# ============================================================

import os
import re
import time
import shutil
import logging
import requests
import numpy as np
import torch
import gradio as gr

from IPython.display import (
    clear_output,
    Javascript,
    display
)


# ============================================================
# TẮT DEBUG ENV
# ============================================================

os.environ["GRADIO_DEBUG"] = "0"


# ============================================================
# PATCH TORCH
# ============================================================

import torch as _torch

if not hasattr(_torch, "_utils"):
    _torch._utils = _torch._C._utils


# ============================================================
# PATCH TRANSFORMERS
# ============================================================

import transformers as _tf


class _SafeAutoFeatureExtractor:

    @staticmethod
    def from_pretrained(model_name, **kwargs):

        try:

            from transformers import AutoConfig

            cfg = AutoConfig.from_pretrained(
                model_name,
                trust_remote_code=True,
                **kwargs
            )

            sr = getattr(
                cfg,
                "sampling_rate",
                24000
            )

        except Exception:

            sr = 24000


        class _Result:
            sampling_rate = sr


        return _Result()


_tf.AutoFeatureExtractor = _SafeAutoFeatureExtractor


# ============================================================
# OMNIVOICE
# ============================================================

from omnivoice import (
    OmniVoice,
    OmniVoiceGenerationConfig
)

from omnivoice.utils.common import get_best_device


# ============================================================
# LOGGING
# ============================================================

logging.basicConfig(
    level=logging.WARNING,
    format="%(asctime)s %(levelname)s: %(message)s"
)

logger = logging.getLogger(__name__)


# ============================================================
# FOLDERS
# ============================================================

VOICE_DIR = "/content/voices"

CUSTOM_VOICE_DIR = "/content/custom_voices"


os.makedirs(
    VOICE_DIR,
    exist_ok=True
)

os.makedirs(
    CUSTOM_VOICE_DIR,
    exist_ok=True
)


# ============================================================
# GITHUB VOICE BASE
# ============================================================

GITHUB_VOICE_BASE = (
    "https://raw.githubusercontent.com/"
    "phong6786789/"
    "OMNIVOICE-MOD/"
    "main/voices/"
)


# ============================================================
# LINK MỞ FULL COLAB
# ============================================================

FULLSCREEN_COLAB_URL = (
    "https://colab.research.google.com/github/"
    "phong6786789/OMNIVOICE-MOD/blob/main/"
    "OmniVoice_CLONE_PhongSubi.ipynb"
    "#scrollTo=9Vfjb9Yb2Qug&fullscreenOutput=true"
)


# ============================================================
# 27 GIỌNG PRESET
# ============================================================

VOICE_DATA = {

    # ========================================================
    # NỮ - 12
    # ========================================================

    "♀ Khánh Huyền":
        "khanhhuyentvc_sample.mp3",

    "♀ Thùy Linh":
        "thuylinh_thuyetminh.mp3",

    "♀ Hoài An":
        "vi_female_hoaian_mb.mp3",

    "♀ Hồng Hạnh":
        "vi_female_honghanh_mn_podcast.mp3",

    "♀ Hồng Ngân":
        "vi_female_hongngan_mn_buon.mp3",

    "♀ Khánh Linh":
        "vi_female_khanhlinh_mb.mp3",

    "♀ Kim Phương":
        "vi_female_kimphuong_mb_tKhLy5k.mp3",

    "♀ Nova":
        "vi_female_nova_default.mp3",

    "♀ Thủy Tiên":
        "vi_female_thuytien_mn.mp3",

    "♀ Thùy Trang":
        "vi_female_thuytrang_mb_rzuuQ7F.mp3",

    "♀ Trâm Anh":
        "vi_female_tramanh_mb_sample.mp3",

    "♀ Trần Anh":
        "vi_female_trananh_mb.mp3",


    # ========================================================
    # NAM - 15
    # ========================================================

    "♂ Đức Trọng":
        "ductrong_sample2.mp3",

    "♂ Đăng Khoa":
        "vi_male_dangkhoa_mb.mp3",

    "♂ Echo":
        "vi_male_echo_default.mp3",

    "♂ Lê Đức":
        "vi_male_leduc_mb_FZBJAUZ.mp3",

    "♂ Lê Hoàng":
        "vi_male_lehoang_mb_SRojRwi.mp3",

    "♂ Lê Nghĩa":
        "vi_male_lenghia_mb_BITWyO7.mp3",

    "♂ Minh Quân":
        "vi_male_minhquan_mb.mp3",

    "♂ Minh Triết":
        "vi_male_minhtriet_mb.mp3",

    "♂ Onyx":
        "vi_male_onyx_default.mp3",

    "♂ Thành Trung":
        "vi_male_thanhtrung_mn_k8K8ONG.mp3",

    "♂ Trí Dũng":
        "vi_male_tridung_mn_sample.mp3",

    "♂ Tuấn Kiệt":
        "vi_male_tuankiet_mn.mp3",

    "♂ Văn Đức":
        "vi_male_vanduc_mn.mp3",

    "♂ Văn Duy":
        "vi_male_vanduy_mb.mp3",

    "♂ Adam":
        "samples_adam.mp3",
}


TOTAL_PRESET_VOICES = len(
    VOICE_DATA
)


# ============================================================
# DOWNLOAD FILE
# ============================================================

def download_file(
    url,
    destination
):

    response = requests.get(
        url,
        timeout=60
    )

    response.raise_for_status()

    with open(
        destination,
        "wb"
    ) as f:

        f.write(
            response.content
        )


# ============================================================
# KIỂM TRA / TẢI 27 GIỌNG
# ============================================================

VOICE_FILES = {}

available_voices = []

failed_voices = []


print("\n========================================")

print(
    f"⬇️ KIỂM TRA / TẢI "
    f"{TOTAL_PRESET_VOICES} GIỌNG"
)

print("========================================")


for voice_name, filename in VOICE_DATA.items():

    local_path = os.path.join(
        VOICE_DIR,
        filename
    )

    raw_url = (
        GITHUB_VOICE_BASE
        +
        filename
    )


    # ========================================================
    # FILE ĐÃ CÓ
    # ========================================================

    if os.path.exists(
        local_path
    ):

        VOICE_FILES[
            voice_name
        ] = local_path

        available_voices.append(
            voice_name
        )

        print(
            f"✅ {voice_name}"
        )

        continue


    # ========================================================
    # DOWNLOAD
    # ========================================================

    try:

        print(
            f"⬇️ {voice_name}"
        )


        download_file(
            raw_url,
            local_path
        )


        if os.path.getsize(
            local_path
        ) < 1000:

            os.remove(
                local_path
            )

            raise RuntimeError(
                "File tải về không hợp lệ."
            )


        VOICE_FILES[
            voice_name
        ] = local_path


        available_voices.append(
            voice_name
        )


        print(
            "   ✅ Hoàn tất"
        )


    except Exception as e:

        failed_voices.append(
            voice_name
        )

        print(
            f"   ❌ {e}"
        )


print("\n========================================")

print(
    f"✅ {len(available_voices)}/"
    f"{TOTAL_PRESET_VOICES} giọng sẵn sàng"
)

print("========================================\n")


# ============================================================
# GPU
# ============================================================

print(
    "🔍 Kiểm tra GPU..."
)


for i in range(30):

    if torch.cuda.is_available():

        print(
            "✅ GPU:",
            torch.cuda.get_device_name(0)
        )

        break

    time.sleep(1)


else:

    print(
        "⚠️ Không phát hiện GPU.\n"
        "Runtime → Change runtime type → T4 GPU"
    )


# ============================================================
# LOAD MODEL
# ============================================================

DEVICE = get_best_device()


MODEL_DTYPE = (

    torch.float16

    if torch.cuda.is_available()

    else torch.float32
)


print(
    "⏳ Đang tải OmniVoice..."
)


model = OmniVoice.from_pretrained(

    "k2-fsa/OmniVoice",

    device_map=DEVICE,

    dtype=MODEL_DTYPE,

    load_asr=True
)


SAMPLING_RATE = (
    model.sampling_rate
)


print(
    f"✅ OmniVoice ready — "
    f"{SAMPLING_RATE} Hz"
)


# ============================================================
# GENERATION CONFIG
# ============================================================

GEN_CFG = OmniVoiceGenerationConfig(

    num_step=32,

    guidance_scale=1.8,

    denoise=True,

    preprocess_prompt=True,

    postprocess_output=True,

    position_temperature=5.0,

    class_temperature=0.2,

    pad_duration=0.1,

    fade_duration=0.1,
)


# ============================================================
# CACHE
# ============================================================

VOICE_PROMPT_CACHE = {}

CUSTOM_VOICE_FILES = {}

CUSTOM_VOICE_PROMPTS = {}

CUSTOM_VOICE_COUNTER = 0


# ============================================================
# DANH SÁCH TOÀN BỘ GIỌNG
# ============================================================

def get_all_voice_names():

    return (

        list(
            VOICE_DATA.keys()
        )

        +

        list(
            CUSTOM_VOICE_FILES.keys()
        )
    )


# ============================================================
# GET VOICE FILE
# ============================================================

def get_voice_file(
    voice_name
):

    # ========================================================
    # CUSTOM VOICE
    # ========================================================

    if voice_name in CUSTOM_VOICE_FILES:

        return CUSTOM_VOICE_FILES[
            voice_name
        ]


    # ========================================================
    # PRESET
    # ========================================================

    if voice_name not in VOICE_DATA:

        return None


    filename = VOICE_DATA[
        voice_name
    ]


    local_path = os.path.join(
        VOICE_DIR,
        filename
    )


    # ========================================================
    # FILE ĐÃ CÓ
    # ========================================================

    if os.path.exists(
        local_path
    ):

        VOICE_FILES[
            voice_name
        ] = local_path


        return local_path


    # ========================================================
    # CHƯA CÓ -> DOWNLOAD
    # ========================================================

    raw_url = (
        GITHUB_VOICE_BASE
        +
        filename
    )


    try:

        download_file(
            raw_url,
            local_path
        )


        if os.path.getsize(
            local_path
        ) < 1000:

            os.remove(
                local_path
            )

            raise RuntimeError(
                "File giọng không hợp lệ."
            )


        VOICE_FILES[
            voice_name
        ] = local_path


        return local_path


    except Exception as e:

        if os.path.exists(
            local_path
        ):

            os.remove(
                local_path
            )


        raise gr.Error(
            f"Không tải được giọng "
            f"{voice_name}:\n{e}"
        )


# ============================================================
# GET VOICE PROMPT
# ============================================================

def get_voice_prompt(
    voice_name
):

    # ========================================================
    # CUSTOM CACHE
    # ========================================================

    if voice_name in CUSTOM_VOICE_PROMPTS:

        return CUSTOM_VOICE_PROMPTS[
            voice_name
        ]


    # ========================================================
    # PRESET CACHE
    # ========================================================

    if voice_name in VOICE_PROMPT_CACHE:

        return VOICE_PROMPT_CACHE[
            voice_name
        ]


    voice_file = get_voice_file(
        voice_name
    )


    if not voice_file:

        raise ValueError(
            "Không tìm thấy giọng."
        )


    voice_prompt = (
        model.create_voice_clone_prompt(
            ref_audio=voice_file
        )
    )


    # ========================================================
    # CACHE
    # ========================================================

    if voice_name in CUSTOM_VOICE_FILES:

        CUSTOM_VOICE_PROMPTS[
            voice_name
        ] = voice_prompt

    else:

        VOICE_PROMPT_CACHE[
            voice_name
        ] = voice_prompt


    return voice_prompt


# ============================================================
# PREVIEW VOICE
# ============================================================

def preview_voice(
    voice_name
):

    if not voice_name:

        return None


    try:

        return get_voice_file(
            voice_name
        )


    except Exception:

        return None


# ============================================================
# NORMALIZE AUDIO
# ============================================================

def normalize_audio(
    audio
):

    if torch.is_tensor(
        audio
    ):

        audio = (
            audio
            .detach()
            .float()
            .cpu()
            .numpy()
        )


    audio = np.asarray(
        audio,
        dtype=np.float32
    )


    audio = np.squeeze(
        audio
    )


    if audio.size == 0:

        return audio


    max_amp = np.max(
        np.abs(
            audio
        )
    )


    if max_amp > 1.0:

        audio = (
            audio
            /
            max_amp
        )


    return audio


# ============================================================
# ĐẾM TỪ + KÝ TỰ
# ============================================================

def count_text(
    text
):

    raw_text = str(
        text
        if text is not None
        else ""
    )


    clean_text = (
        raw_text.strip()
    )


    if clean_text:

        words = len(
            re.findall(
                r"\S+",
                clean_text
            )
        )

    else:

        words = 0


    chars = len(
        raw_text
    )


    return (
        f"{words:,} từ"
        f"  •  "
        f"{chars:,} ký tự"
    )


# ============================================================
# CLONE GIỌNG
# ============================================================

def create_custom_voice(
    audio_file,
    voice_name,
    progress=gr.Progress()
):

    global CUSTOM_VOICE_COUNTER


    # ========================================================
    # CHECK AUDIO
    # ========================================================

    if not audio_file:

        raise gr.Error(
            "Hãy upload hoặc thu âm giọng mẫu."
        )


    if not os.path.exists(
        audio_file
    ):

        raise gr.Error(
            "Không tìm thấy file âm thanh."
        )


    progress(
        0.10,
        desc="🎤 Đang chuẩn bị..."
    )


    # ========================================================
    # VOICE NAME
    # ========================================================

    if voice_name:

        voice_name = (
            voice_name.strip()
        )

    else:

        voice_name = (
            "Giọng của tôi"
        )


    if not voice_name:

        voice_name = (
            "Giọng của tôi"
        )


    CUSTOM_VOICE_COUNTER += 1


    display_name = (
        f"🎤 {voice_name} "
        f"#{CUSTOM_VOICE_COUNTER}"
    )


    # ========================================================
    # EXTENSION
    # ========================================================

    extension = os.path.splitext(
        audio_file
    )[1]


    if not extension:

        extension = ".wav"


    # ========================================================
    # DESTINATION
    # ========================================================

    destination = os.path.join(

        CUSTOM_VOICE_DIR,

        (
            f"custom_voice_"
            f"{CUSTOM_VOICE_COUNTER}"
            f"{extension}"
        )
    )


    # ========================================================
    # COPY
    # ========================================================

    shutil.copy2(
        audio_file,
        destination
    )


    # ========================================================
    # CREATE CLONE PROMPT
    # ========================================================

    progress(
        0.40,
        desc="🧬 Đang clone giọng..."
    )


    try:

        voice_prompt = (
            model.create_voice_clone_prompt(
                ref_audio=destination
            )
        )


    except Exception as e:

        if os.path.exists(
            destination
        ):

            os.remove(
                destination
            )


        raise gr.Error(
            f"Không thể clone giọng:\n{e}"
        )


    # ========================================================
    # SAVE
    # ========================================================

    CUSTOM_VOICE_FILES[
        display_name
    ] = destination


    CUSTOM_VOICE_PROMPTS[
        display_name
    ] = voice_prompt


    progress(
        1.0,
        desc="✅ Clone hoàn tất"
    )


    # ========================================================
    # UPDATE DROPDOWN
    # ========================================================

    return (

        gr.Dropdown(
            choices=get_all_voice_names(),
            value=display_name,
            filterable=False,
            allow_custom_value=False
        ),

        destination,

        (
            f"✅ Đã tạo giọng "
            f"**{display_name}**"
        )
    )


# ============================================================
# GENERATE VOICE
# ============================================================

def generate_voice(
    text,
    voice_name,
    speed,
    pause_duration,
    progress=gr.Progress()
):

    # ========================================================
    # TEXT
    # ========================================================

    if not text:

        raise gr.Error(
            "Bạn chưa nhập nội dung."
        )


    text = (
        text.strip()
    )


    if not text:

        raise gr.Error(
            "Bạn chưa nhập nội dung."
        )


    # ========================================================
    # VOICE
    # ========================================================

    if not voice_name:

        raise gr.Error(
            "Bạn chưa chọn giọng."
        )


    if voice_name not in get_all_voice_names():

        raise gr.Error(
            "Giọng không hợp lệ."
        )


    # ========================================================
    # PREPARE VOICE
    # ========================================================

    progress(
        0.03,
        desc="🎤 Đang chuẩn bị giọng..."
    )


    try:

        voice_prompt = (
            get_voice_prompt(
                voice_name
            )
        )


    except Exception as e:

        raise gr.Error(
            f"Lỗi xử lý giọng:\n{e}"
        )


    # ========================================================
    # CHIA ĐOẠN
    # ========================================================

    paragraphs = [

        p.strip()

        for p in re.split(
            r"\n\s*\n",
            text
        )

        if p.strip()
    ]


    if not paragraphs:

        raise gr.Error(
            "Nội dung không hợp lệ."
        )


    total = len(
        paragraphs
    )


    all_audio = []


    # ========================================================
    # GENERATE
    # ========================================================

    for i, paragraph in enumerate(
        paragraphs
    ):


        progress(

            (
                0.08
                +
                0.84
                *
                (
                    i
                    /
                    total
                )
            ),

            desc=(
                f"🎙️ Đang tạo đoạn "
                f"{i + 1}/{total}..."
            )
        )


        try:

            result = model.generate(

                text=paragraph,

                voice_clone_prompt=voice_prompt,

                # =================================================
                # CHỈ TIẾNG VIỆT
                # =================================================
                language="vi",

                speed=float(
                    speed
                ),

                generation_config=GEN_CFG
            )


        except Exception as e:

            raise gr.Error(
                f"Lỗi đoạn "
                f"{i + 1}:\n{e}"
            )


        # ====================================================
        # AUDIO
        # ====================================================

        audio = normalize_audio(
            result[0]
        )


        if audio.size == 0:

            raise gr.Error(
                f"Đoạn {i + 1} "
                f"không tạo được audio."
            )


        all_audio.append(
            audio
        )


        # ====================================================
        # PAUSE
        # ====================================================

        if i < total - 1:

            silence_length = int(

                SAMPLING_RATE
                *
                float(
                    pause_duration
                )
            )


            if silence_length > 0:

                silence = np.zeros(

                    silence_length,

                    dtype=np.float32
                )


                all_audio.append(
                    silence
                )


    # ========================================================
    # GHÉP AUDIO
    # ========================================================

    progress(
        0.95,
        desc="🎧 Đang hoàn thiện audio..."
    )


    final_audio = np.concatenate(
        all_audio
    )


    final_audio = normalize_audio(
        final_audio
    )


    waveform = (

        final_audio
        *
        32767

    ).astype(
        np.int16
    )


    progress(
        1.0,
        desc="✅ Hoàn tất"
    )


    return (
        SAMPLING_RATE,
        waveform
    )


# ============================================================
# DEFAULT VOICE
# ============================================================

DEFAULT_VOICE = (
    "♂ Adam"
)


if DEFAULT_VOICE not in VOICE_DATA:

    DEFAULT_VOICE = (
        list(
            VOICE_DATA.keys()
        )[0]
    )


# ============================================================
# DEFAULT PREVIEW
# ============================================================

try:

    DEFAULT_PREVIEW = (
        get_voice_file(
            DEFAULT_VOICE
        )
    )


except Exception:

    DEFAULT_PREVIEW = None


# ============================================================
# CSS
# ============================================================

CSS = """

/* ==========================================================
   MAIN APP
========================================================== */

.gradio-container {

    max-width: 1100px !important;

    width: 96% !important;

    margin: 0 auto !important;

    padding:
        18px
        24px
        45px
        24px !important;
}


/* ==========================================================
   HEADER
========================================================== */

#app_header {

    margin-bottom: 4px !important;
}


#app_header h1 {

    font-size: 30px !important;

    margin-bottom: 4px !important;
}


/* ==========================================================
   FULL TAB AREA
========================================================== */

#full_tab_area {

    margin:
        8px
        0
        16px
        0 !important;
}


/* ==========================================================
   FULL TAB BUTTON
========================================================== */

.full-tab-btn {

    display: flex !important;

    align-items: center !important;

    justify-content: center !important;

    width: 100% !important;

    min-height: 52px !important;

    box-sizing: border-box !important;

    border-radius: 12px !important;

    background: #6366f1 !important;

    color: white !important;

    text-decoration: none !important;

    font-size: 16px !important;

    font-weight: 700 !important;

    cursor: pointer !important;

    transition:
        opacity 0.15s ease,
        transform 0.15s ease !important;
}


.full-tab-btn:hover {

    opacity: 0.92 !important;

    transform: translateY(-1px) !important;
}


/* ==========================================================
   TEXT AREA
========================================================== */

#main_text textarea {

    min-height: 290px !important;

    font-size: 17px !important;

    line-height: 1.65 !important;

    padding: 16px !important;
}


/* ==========================================================
   TEXT COUNTER
========================================================== */

#text_counter {

    margin-top: -5px !important;

    margin-bottom: 8px !important;
}


#text_counter input {

    border: none !important;

    background: transparent !important;

    box-shadow: none !important;

    padding-left: 2px !important;

    font-size: 14px !important;

    font-weight: 500 !important;

    opacity: 0.72 !important;
}


/* ==========================================================
   GENERATE BUTTON
========================================================== */

#generate_btn button {

    min-height: 58px !important;

    font-size: 18px !important;

    font-weight: 700 !important;
}


/* ==========================================================
   FOOTER
========================================================== */

footer {

    display: none !important;
}


/* ==========================================================
   MOBILE
========================================================== */

@media (max-width: 700px) {

    .gradio-container {

        width: 100% !important;

        padding: 12px !important;
    }


    #app_header h1 {

        font-size: 24px !important;
    }


    #main_text textarea {

        min-height: 235px !important;

        font-size: 16px !important;
    }

}

"""


# ============================================================
# THEME
# ============================================================

THEME = gr.themes.Soft(
    primary_hue="indigo"
)


# ============================================================
# CLOSE OLD APP
# ============================================================

gr.close_all()


# ============================================================
# BUILD UI
# ============================================================

with gr.Blocks(
    title="Phong Subi - OMNIVOICE MOD"
) as demo:


    # ========================================================
    # HEADER
    # ========================================================

    gr.Markdown(
        f"""
# 🎙️ Phong Subi - OMNIVOICE MOD

**🇻🇳 {TOTAL_PRESET_VOICES} giọng Tiếng Việt • Clone giọng riêng**
""",
        elem_id="app_header"
    )


    # ========================================================
    # CHỈ 1 NÚT MỞ FULL TAB
    # ========================================================

    gr.HTML(
        f"""
        <div id="full_tab_area">

            <a
                class="full-tab-btn"
                href="{FULLSCREEN_COLAB_URL}"
                target="_blank"
                rel="noopener noreferrer"
            >
                ↗ &nbsp; MỞ FULL Ở TAB MỚI
            </a>

        </div>
        """
    )


    # ========================================================
    # VOICE
    # ========================================================

    with gr.Group():


        gr.Markdown(
            "### 🎤 Giọng đọc"
        )


        voice_selector = gr.Dropdown(

            choices=list(
                VOICE_DATA.keys()
            ),

            value=DEFAULT_VOICE,

            label="Chọn giọng",

            interactive=True,

            # Không cho gõ vào dropdown
            filterable=False,

            # Không cho custom value
            allow_custom_value=False
        )


        # ====================================================
        # PREVIEW
        # ====================================================

        preview_audio = gr.Audio(

            value=DEFAULT_PREVIEW,

            label="🔊 Nghe thử giọng",

            interactive=False,

            autoplay=False
        )


        # ====================================================
        # CLONE VOICE
        # ====================================================

        with gr.Accordion(

            "➕ Clone giọng của bạn",

            open=False

        ):


            gr.Markdown(
                """
Bạn có thể **upload file âm thanh** hoặc **thu âm trực tiếp**.

Để clone tốt hơn:

- Chỉ một người nói
- Không nhạc nền
- Ít tiếng ồn
- Giọng rõ ràng

Chỉ clone giọng mà bạn có quyền sử dụng.
"""
            )


            custom_voice_name = gr.Textbox(

                label="Tên giọng",

                placeholder=(
                    "Ví dụ: Giọng của tôi"
                )
            )


            custom_audio = gr.Audio(

                sources=[
                    "upload",
                    "microphone"
                ],

                type="filepath",

                label=(
                    "🎤 Upload hoặc "
                    "thu âm giọng mẫu"
                )
            )


            clone_button = gr.Button(

                "✨ TẠO GIỌNG CLONE",

                variant="secondary"
            )


            clone_status = gr.Markdown()


    # ========================================================
    # TEXT
    # ========================================================

    with gr.Group():


        gr.Markdown(
            "### 📝 Nội dung"
        )


        text_input = gr.Textbox(

            lines=12,

            label="",

            placeholder=(
                "Nhập nội dung Tiếng Việt tại đây...\n\n"
                "Xuống dòng 2 lần nếu muốn "
                "tạo khoảng nghỉ giữa các đoạn."
            ),

            elem_id="main_text"
        )


        # ====================================================
        # COUNTER
        # ====================================================

        text_counter = gr.Textbox(

            value="0 từ  •  0 ký tự",

            label="",

            interactive=False,

            container=False,

            elem_id="text_counter"
        )


        # ====================================================
        # SETTINGS
        # ====================================================

        with gr.Row():


            speed_slider = gr.Slider(

                minimum=0.70,

                maximum=1.30,

                value=0.95,

                step=0.05,

                label="⚡ Tốc độ đọc"
            )


            pause_slider = gr.Slider(

                minimum=0,

                maximum=2.0,

                value=0.3,

                step=0.1,

                label=(
                    "⏸ Nghỉ giữa đoạn "
                    "(giây)"
                )
            )


        # ====================================================
        # GENERATE BUTTON
        # ====================================================

        generate_button = gr.Button(

            "🎙️ TẠO GIỌNG NÓI",

            variant="primary",

            elem_id="generate_btn"
        )


    # ========================================================
    # OUTPUT
    # ========================================================

    with gr.Group():


        gr.Markdown(
            "### 🎧 Kết quả"
        )


        output_audio = gr.Audio(

            label="",

            autoplay=False
        )


    # ========================================================
    # EVENTS
    # ========================================================


    # ========================================================
    # ĐẾM TỪ NGAY KHI GÕ
    # ========================================================

    text_input.input(

        fn=count_text,

        inputs=text_input,

        outputs=text_counter,

        queue=False,

        show_progress="hidden"
    )


    # ========================================================
    # BACKUP KHI PASTE / CHANGE
    # ========================================================

    text_input.change(

        fn=count_text,

        inputs=text_input,

        outputs=text_counter,

        queue=False,

        show_progress="hidden"
    )


    # ========================================================
    # ĐỔI GIỌNG -> PREVIEW
    # ========================================================

    voice_selector.change(

        fn=preview_voice,

        inputs=voice_selector,

        outputs=preview_audio,

        queue=False,

        show_progress="hidden"
    )


    # ========================================================
    # CLONE
    # ========================================================

    clone_button.click(

        fn=create_custom_voice,

        inputs=[
            custom_audio,
            custom_voice_name
        ],

        outputs=[
            voice_selector,
            preview_audio,
            clone_status
        ],

        concurrency_limit=1,

        show_progress="minimal"
    )


    # ========================================================
    # GENERATE
    # ========================================================

    generate_button.click(

        fn=generate_voice,

        inputs=[
            text_input,
            voice_selector,
            speed_slider,
            pause_slider
        ],

        outputs=output_audio,

        concurrency_limit=1,

        show_progress="full"
    )


# ============================================================
# QUEUE
# ============================================================

demo.queue(
    default_concurrency_limit=1
)


# ============================================================
# CLEAR LOG
# ============================================================

clear_output(
    wait=True
)


print(
    "✅ Phong Subi - OMNIVOICE MOD đã sẵn sàng"
)


# ============================================================
# TĂNG CHIỀU CAO OUTPUT COLAB
# ============================================================

try:

    display(
        Javascript(
            """
            if (
                typeof google !== "undefined" &&
                google.colab &&
                google.colab.output
            ) {

                google.colab.output.setIframeHeight(
                    0,
                    true,
                    {
                        maxHeight: 6000
                    }
                );

            }
            """
        )
    )

except Exception:

    pass


# ============================================================
# LAUNCH
# ============================================================

demo.launch(

    server_name="0.0.0.0",

    share=True,

    inline=True,

    # Không debug chạy mãi
    debug=False,

    # Vẫn hiện lỗi trên UI
    show_error=True,

    # Giảm log Gradio
    quiet=True,

    theme=THEME,

    css=CSS
)


# ============================================================
# AUTO SCROLL + CHUÔNG TING TING
# ============================================================

try:

    display(
        Javascript(
            """
            setTimeout(async () => {

                // ============================================
                // AUTO SCROLL XUỐNG GIAO DIỆN
                // ============================================

                try {

                    const cells =
                        document.querySelectorAll(
                            "colab-code-cell"
                        );


                    if (
                        cells &&
                        cells.length > 0
                    ) {

                        const currentCell =
                            cells[
                                cells.length - 1
                            ];


                        currentCell.scrollIntoView({

                            behavior:
                                "smooth",

                            block:
                                "start"

                        });

                    }

                }

                catch (e) {

                    console.log(
                        "Auto scroll error:",
                        e
                    );

                }


                // ============================================
                // TING TING
                // ============================================

                try {

                    const AudioContextClass =
                        window.AudioContext
                        ||
                        window.webkitAudioContext;


                    if (
                        !AudioContextClass
                    ) {

                        return;

                    }


                    const ctx =
                        new AudioContextClass();


                    // ========================================
                    // RESUME AUDIO CONTEXT
                    // ========================================

                    if (
                        ctx.state ===
                        "suspended"
                    ) {

                        try {

                            await ctx.resume();

                        }

                        catch (e) {

                        }

                    }


                    // ========================================
                    # FUNCTION TING
                    // ========================================

                    function ting(
                        startTime,
                        frequency
                    ) {

                        const oscillator =
                            ctx.createOscillator();


                        const gain =
                            ctx.createGain();


                        oscillator.connect(
                            gain
                        );


                        gain.connect(
                            ctx.destination
                        );


                        oscillator.type =
                            "sine";


                        oscillator.frequency.setValueAtTime(
                            frequency,
                            startTime
                        );


                        gain.gain.setValueAtTime(
                            0.0001,
                            startTime
                        );


                        gain.gain.exponentialRampToValueAtTime(
                            0.16,
                            startTime + 0.015
                        );


                        gain.gain.exponentialRampToValueAtTime(
                            0.0001,
                            startTime + 0.25
                        );


                        oscillator.start(
                            startTime
                        );


                        oscillator.stop(
                            startTime + 0.28
                        );

                    }


                    // ========================================
                    // TING 1
                    // ========================================

                    ting(
                        ctx.currentTime + 0.05,
                        880
                    );


                    // ========================================
                    // TING 2
                    // ========================================

                    ting(
                        ctx.currentTime + 0.32,
                        1175
                    );


                }

                catch (e) {

                    console.log(
                        "Ting error:",
                        e
                    );

                }


            }, 900);
            """
        )
    )

except Exception:

    pass

✅ Phong Subi - OMNIVOICE MOD đã sẵn sàng


<IPython.core.display.Javascript object>

* Running on public URL: https://4f7dcaee1748a34e70.gradio.live


<IPython.core.display.Javascript object>